<img src="../../../At-Home-Round/Chameleon/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), manche a domicile](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/At-Home-Round/Chameleon/Chameleon.ipynb)

# Chameleon

**Chameleon** est un jeu de devinette de mots dans lequel les joueurs communiquent des idees a l'aide d'icones visuelles. Il existe deux roles : le **donneur d'indices** et le **devineur**. Les deux joueurs disposent d'un ensemble commun d'icones visuelles, chacune accompagnee d'une description connue. Voici quelques icones d'exemple :

<img src="../../../At-Home-Round/Chameleon/figs/Chameleon Fig 1.png" width="300">

Le donneur d'indices choisit d'abord un **secret**, un mot ou une expression, puis fournit un **indice** en pointant une **sequence ordonnee** d'icones issues de l'ensemble partage : il est interdit de parler ou d'ecrire.

L'ordre des icones est significatif :
- La **premiere icone** represente generalement l'idee centrale du secret.
- Les **icones suivantes** apportent un contexte qui aide a preciser ou enrichir le concept principal.

### Exemple 1

L'indice suivant peut etre interprete comme *un lieu ou se deroule un metier qui combat le feu*, autrement dit une **caserne de pompiers** :

<img src="../../../At-Home-Round/Chameleon/figs/Chameleon Fig 2.png" width="300">

Si l'ordre des icones est inverse, l'indice peut suggerer *un metier qui combat le feu dans une maison*, ce qui renvoie a un **pompier** :

<img src="../../../At-Home-Round/Chameleon/figs/Chameleon Fig 3.png" width="300">

### Exemple 2

Une meme icone peut prendre des sens differents selon le contexte. Par exemple, l'icone coeur peut apparaitre dans un indice qui suggere *un outil utilise par les medecins pour ecouter le coeur*, soit un **stethoscope** :

<img src="../../../At-Home-Round/Chameleon/figs/Chameleon Fig 4.png" width="300">

La meme icone coeur peut au contraire apparaitre dans un indice qui suggere *un personnage fictif a la fois mort et vivant*, c'est-a-dire un **zombie** :

<img src="../../../At-Home-Round/Chameleon/figs/Chameleon Fig 5.png" width="300">

## Tache

Votre tache est de construire un programme d'IA qui joue le role du **devineur** : a partir d'un indice (la sequence ordonnee d'icones choisie par le donneur), votre programme doit identifier le secret.

Pour aider votre programme, un ensemble complet de **choix** possibles est fourni : il est garanti que le bon secret y figure. Votre programme peut produire une **liste ordonnee d'au plus 10 propositions**. Vous obtenez un score si le bon secret apparait dans la liste, avec un score d'autant plus eleve qu'il apparait **tot**.

Pour rendre le defi plus interessant, deux contraintes cles :

- Votre solution doit utiliser un modele de **moins d'1 milliard de parametres** (environ 4 Go de memoire).
- Elle **ne doit pas** dependre d'APIs de modeles externes au moment de l'inference.

Vous pouvez utiliser des modeles externes pre-entraines pendant le developpement et l'entrainement, mais votre **solution soumise doit etre autonome** et respecter la contrainte de taille.

## Implementation

Implementez la fonction `guess_words(hint, choices)` qui renvoie une liste de exactement **10 propositions** classees de la plus probable a la moins probable.

- **`hint`** : une liste d'entiers representant la sequence ordonnee d'icones pointees par le donneur d'indices.
- **`choices`** : une liste de mots ou expressions candidats pour le secret. Le bon secret y figure forcement.

### Liste d'icones

Vous disposez d'une liste d'icones, chacune associee a sa description :

```
!pip install -U datasets
```
```python
from IPython.display import display
from PIL import Image
from datasets import load_dataset

hint_description = load_dataset("JettChenT/ioai-chameleon-hint_descriptions")['train']

hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

# exemple
display(hint_description[7]['icons'])
print(hint_description[7]['description'])
```
```
Flora
Plant
Nature
```
<img src="../../../At-Home-Round/Chameleon/figs/Chameleon Fig 6.png" width="100">


### Donnees de validation

Vous pouvez acceder aux donnees de validation comme suit :

```
validation_data = load_dataset("JettChenT/ioai-chameleon-takehome_validation")['validation']

# un exemple de partie
print(validation_data[0])
```
```
{'hints': [6, 61, 63, 115, 33], 'options': ['sunflower', 'credit card', 'dinosaur', 'key', 'sundial', 'lawyer', 'doorbell', 'trash can', 'crab', 'xylophone', 'queen', 'ambulance', 'space station', 'wallet', 'market', 'orchestra', 'chocolate', 'zipper', 'rhinoceros', 'fashion', 'butterfly', 'truck', 'palm tree', 'cake', 'radio', 'seal', 'mailbox', 'magnifying glass', 'prison', 'polar bear', 'mouse', 'alumunium foil', 'harmonica', 'shell', 'boxers', 'tricycle', 'peacock', 'kettle', 'mountain', 'harbor', 'coffee', 'fireworks', 'pie', 'gravity', 'teacher', 'museum', 'bedroom', 'robe', 'sunscreen', 'robot', 'piano', 'baker', 'plankton', 'scarf', 'bee', 'mosquito', 'accountant', 'umbrella', 'janitor', 'thief', 'parrot', 'koala', 'refrigerator', 'drone', 'dining room', 'soap', 'whistle', 'bicycle', 'train tracks', 'penguin', 'octopus', 'hula hoop', 'ice skates', 'nightmare', 'diving suit', 'horseshoe', 'dynamite', 'surfboard', 'toaster', 'gloves', 'broom', 'postal worker', 'lipstick', 'sewing machine', 'salad', 'dam', 'pool', 'fertilizer', 'shovel', 'speaker', 'seahorse', 'submarine', 'pig', 'mango', 'fire station', 'ping-pong', 'hotel', 'carpet', 'shoes', 'parachute'], 'label': 'seal'}
```

### Exemple d'implementation

Voici un exemple d'implementation de la fonction `guess_words`.

```python
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode([
    'hello world',
    'fun and games'
])

print(f"Embedding shape: {embeddings.shape}")
```
```python
def hints_to_sentence(hints: list[int]) -> str:
  sentence = "The following hints at our target word:\n<HINT_PRIMARY>\n"
  for i, hint in enumerate(hints):
    sentence += f"{hint_description[hint]['description']}"
    if i == 0:
      sentence += "\n</HINT_PRIMARY>\n<HINT>\n"
    elif i < len(hints) - 1:
      sentence += "\n</HINT>\n<HINT>\n"
    else:
      sentence += "\n</HINT>"
  return sentence

def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

print(hints_to_sentence([1, 2, 3]))
```

```python
def find_most_similar(query, sentences, model, top_k=10):
    # Encode la requete et les phrases
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calcule les similarites
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Top-k
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results
```
### Fine-tuning basique
```python
# Fine-tune le modele

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

import os
os.environ["WANDB_DISABLED"] = "true"

train_examples = []
for val in validation_data:
  train_examples.append(InputExample(texts=[hints_to_sentence(val['hints']), choice_to_doc(val['label'])], label=1))

# Cree le DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)

# Definit la fonction de perte
train_loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=5,
    output_path='./model',
    optimizer_params={'lr': 1e-6},
    weight_decay=0.01,
    save_best_model=True,
    show_progress_bar=True
)
```

```
# Pour l'inference, on charge simplement le modele depuis le dossier ./custom-model

ft_model_loaded = SentenceTransformer('./model')
```
```
def guess_words(hints: list[int], choices: list[str]) -> list[str]:
  query = hints_to_sentence(hints)
  results = find_most_similar(query, choices, ft_model_loaded)
  return [result['sentence'] for result in results]
```

## Evaluation

Une soumission est consideree comme correcte si le secret apparait parmi les 10 propositions renvoyees par votre systeme. C'est evalue avec la metrique **Hits@10**.

Pour recompenser les propositions qui classent la bonne reponse plus haut, on utilise aussi la metrique **NDCG@10** (Normalized Discounted Cumulative Gain). Le score final est une combinaison ponderee : **90 % Hits@10** + **10 % NDCG@10**.

### Hits@10

- **Hits@10 = 1** si la bonne reponse figure parmi les 10 propositions ;
- **Hits@10 = 0** sinon.

### NDCG@10

Donne un credit partiel selon le rang de la bonne reponse. Si elle apparait au rang *i* (en commencant a 1) :

$$
\text{NDCG@10} = \frac{1}{\log_2(i + 1)}
$$

**Exemples :**
- Rang 1 -> 1.00
- Rang 2 -> ~0.63
- Rang 4 -> ~0.43
- Rang 10 -> ~0.29

### Score final

$$
\text{Score final} = 0.9 \times \text{Hits@10} + 0.1 \times \text{NDCG@10}
$$

La fonction suivante calcule votre score :

```python
import math

def score(guesses: list[str], gold: str):
    # Normalise en minuscules
    guesses = [g.lower() for g in guesses[:10]]
    gold = gold.lower()

    result = {
        "hits@10": 0.0,
        "ndcg@10": 0.0,
        "total_score": 0.0
    }

    if gold in guesses:
        rank = guesses.index(gold)
        result["hits@10"] = 1.0
        result["ndcg@10"] = 1.0 / math.log2(rank + 2)  # rank + 2 car l'index commence a 0
    else:
        result["hits@10"] = 0.0
        result["ndcg@10"] = 0.0

    result["total_score"] = 0.9 * result["hits@10"] + 0.1 * result["ndcg@10"]
    return result

print(score(['cat', 'dog', 'tree', 'flower', 'rock', 'water', 'fried rice', 'airplane', 'cactus', 'tiger'], gold='cactus'))
```


## Chargement des donnees


In [ ]:
from IPython.display import display
from PIL import Image
from datasets import Dataset
TRAIN_TEXT = "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

# exemple
display(hint_description[7]['icons'])
print(hint_description[7]['description'])

In [ ]:
validation_data = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/takehome_validation")
validation_data

## Implementer le devineur de mots-cles


L'acces a Internet est autorise sur Bohrium, ce qui permet aux concurrents de telecharger des modeles pre-entraines depuis Hugging Face. Cependant, les serveurs etant en Chine continentale, ils sont soumis a des restrictions reseau qui bloquent l'acces a des sources comme huggingface.co. Pour contourner cela, on peut utiliser des miroirs heberges (qui seront egalement fournis lors de la manche sur site). Pour Hugging Face, definissez `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'` pour utiliser un miroir Hugging Face accessible depuis le serveur de Bohrium.


In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('/all-MiniLM-L6-v2')
embeddings = model.encode([
    'hello world',
    'fun and games'
])

print(f"Embedding shape: {embeddings.shape}")

In [ ]:
# Enregistre le modele dans /personal sur Bohrium
model.save('mymodel1')
!cp mymodel1 /personal

In [ ]:
def hints_to_sentence(hints: list[int]) -> str:
  sentence = "The following hints at our target word:\n<HINT_PRIMARY>\n"
  for i, hint in enumerate(hints):
    sentence += f"{hint_description[hint]['description']}"
    if i == 0:
      sentence += "\n</HINT_PRIMARY>\n<HINT>\n"
    elif i < len(hints) - 1:
      sentence += "\n</HINT>\n<HINT>\n"
    else:
      sentence += "\n</HINT>"
  return sentence

def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

print(hints_to_sentence([1, 2, 3]))

In [ ]:
# Fine-tune le modele

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

import os
os.environ["WANDB_DISABLED"] = "true"

train_examples = []
for val in validation_data:
  train_examples.append(InputExample(texts=[hints_to_sentence(val['hints']), choice_to_doc(val['label'])], label=1))

# Cree le DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)

# Definit la fonction de perte
train_loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=5,
    output_path='./model',
    optimizer_params={'lr': 1e-6},
    weight_decay=0.01,
    save_best_model=True,
    show_progress_bar=True
)

In [ ]:
ft_model_loaded = SentenceTransformer("./model") # Charge le modele fine-tune

def find_most_similar(query, sentences, model, top_k=10):
    # Encode la requete et les phrases
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calcule les similarites
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Recupere les top-k plus similaires
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results

def guess_words(hints: list[int], choices: list[str]) -> list[str]:
  query = hints_to_sentence(hints)
  results = find_most_similar(query, choices, ft_model_loaded)
  return [result['sentence'] for result in results]

In [ ]:
import math

def score(guesses: list[str], gold: str):
    # Normalise en minuscules
    guesses = [g.lower() for g in guesses[:10]]
    gold = gold.lower()

    result = {
        "hits@10": 0.0,
        "ndcg@10": 0.0,
        "total_score": 0.0
    }

    if gold in guesses:
        rank = guesses.index(gold)
        result["hits@10"] = 1.0
        result["ndcg@10"] = 1.0 / math.log2(rank + 2)  # rank + 2 car l'index commence a 0
    else:
        result["hits@10"] = 0.0
        result["ndcg@10"] = 0.0

    result["total_score"] = 0.9 * result["hits@10"] + 0.1 * result["ndcg@10"]
    return result

print(score(['cat', 'dog', 'tree', 'flower', 'rock', 'water', 'fried rice', 'airplane', 'cactus', 'tiger'], gold='cactus'))

In [ ]:
from tqdm.notebook import tqdm

# score sur l'ensemble de validation
guesses = []
total_scores = 0.0
for example in tqdm(validation_data):
    guesses.append(guess_words(example['hints'], example['options']))

    total_scores += score(guesses[-1], example['label'])['total_score']


print(f"Average validation score: {total_scores / len(validation_data)}")

## Soumission du modele


In [ ]:
model_code = """
from sentence_transformers import SentenceTransformer
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

TRAIN_TEXT = "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

model = SentenceTransformer("./model")

def hints_to_sentence(hints: list[int]) -> str:
  sentence = "The following hints at our target word:\\n<HINT_PRIMARY>\\n"
  for i, hint in enumerate(hints):
    sentence += f"{hint_description[hint]['description']}"
    if i == 0:
      sentence += "\\n</HINT_PRIMARY>\\n<HINT>\\n"
    elif i < len(hints) - 1:
      sentence += "\\n</HINT>\\n<HINT>\\n"
    else:
      sentence += "\\n</HINT>"
  return sentence

def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

def find_most_similar(query, sentences, model, top_k=10):
    # Encode la requete et les phrases
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calcule les similarites
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Recupere les top-k plus similaires
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results

def guess_words(hints: list[int], choices: list[str]) -> list[str]:
  query = hints_to_sentence(hints)
  results = find_most_similar(query, choices, model)
  return [result['sentence'] for result in results]
"""

with open("submission_model.py", "w") as f:
  f.write(model_code)

print("Code d'inference ecrit dans submission_model.py")

In [ ]:
import shutil
import os
import tempfile

# Cree un repertoire temporaire avec la structure souhaitee
with tempfile.TemporaryDirectory() as temp_dir:
    # Copie les fichiers dans le repertoire temporaire
    shutil.copy('submission_model.py', temp_dir)
    shutil.copytree('./model', os.path.join(temp_dir, 'model'))
    
    # Cree l'archive zip
    shutil.make_archive('submission', 'zip', temp_dir)